# Follica: Photo-Based Scalp Disease Classifier (Colab Training)

Trains a MobileNetV4 classifier on the Kaggle "Hair Diseases" dataset (10 classes, 12,000 images)
and exports `follica_mobilenetv4.onnx` for the backend.

How to run:
1. Runtime > Change runtime type > T4 GPU.
2. Upload `kaggle.zip` (the dataset) to the root of your Google Drive (My Drive).
3. Runtime > Run all. Approve the Drive access prompt.
4. Download `follica_mobilenetv4.onnx` and `follica_classes.json` from Drive/follica_output.

The reported accuracy is measured on the held-out test split only, using the checkpoint
chosen on the validation split. Report exactly what is printed; do not edit it.

In [ ]:
# 1. Hardware Verification & Package Installation
!nvidia-smi
!pip install -q timm onnx onnxscript onnxruntime albumentations pillow scikit-learn matplotlib tqdm

In [ ]:
# 2. Mount Drive and unpack the dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
OUT_DIR = '/content/drive/MyDrive/follica_output'
os.makedirs(OUT_DIR, exist_ok=True)

ZIP_PATH = '/content/drive/MyDrive/kaggle.zip'
DATA_DIR = '/content/data'
assert os.path.exists(ZIP_PATH), 'Upload kaggle.zip to the root of My Drive first.'
if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

ROOT = None
for d, dirs, _ in os.walk(DATA_DIR):
    if {'train', 'val', 'test'} <= set(dirs):
        ROOT = d
        break
assert ROOT, 'Could not find train/val/test folders inside the zip.'
CLASSES = sorted(os.listdir(os.path.join(ROOT, 'train')))
print('Dataset root:', ROOT)
print('Classes (%d):' % len(CLASSES), CLASSES)

In [ ]:
# 3. Dataset audit: counts and leakage check (perceptual hash across splits)
import glob, collections
import numpy as np
from PIL import Image

def list_split(split):
    paths, labels = [], []
    for i, c in enumerate(CLASSES):
        for p in sorted(glob.glob(os.path.join(ROOT, split, c, '*'))):
            paths.append(p); labels.append(i)
    return paths, labels

splits = {s: list_split(s) for s in ('train', 'val', 'test')}
for s, (p, l) in splits.items():
    print(s, len(p), dict(collections.Counter(CLASSES[i] for i in l)))

def ahash(path):
    im = Image.open(path).convert('L').resize((16, 16))
    a = np.asarray(im, dtype=np.float32)
    return ''.join('1' if v else '0' for v in (a > a.mean()).flatten())

hashes = {s: {ahash(p) for p in splits[s][0]} for s in ('train', 'val', 'test')}
leak_val = len(hashes['train'] & hashes['val'])
leak_test = len(hashes['train'] & hashes['test'])
print('Near-duplicate hashes shared train/val:', leak_val)
print('Near-duplicate hashes shared train/test:', leak_test)
if leak_test > 0:
    print('WARNING: possible train/test leakage. Accuracy may be inflated; state this in the report.')

In [ ]:
# 4. Optical Augmentations Tailored for Smartphone Cameras
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np

train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=30, p=0.5),
    # Simulates optical phone camera sensor variations:
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.5),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])
print('Data augmentation pipeline configured.')

In [ ]:
# 5. Model
import torch, torch.nn as nn, timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = len(CLASSES)
try:
    model = timm.create_model('mobilenetv4_conv_small.e1200_r224_in1k', pretrained=True, num_classes=NUM_CLASSES)
    ARCH = 'mobilenetv4_conv_small'
except Exception as e:
    print('MobileNetV4 unavailable, falling back to EfficientNet-B0:', e)
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=NUM_CLASSES)
    ARCH = 'efficientnet_b0'
model = model.to(device)
print(ARCH, 'on', device, '| trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# 6. Training (AdamW + cosine schedule, mixed precision, best-val checkpoint saved to Drive)
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

class ScalpDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert('RGB'))
        return self.transform(image=img)['image'], torch.tensor(self.labels[i], dtype=torch.long)

train_dl = DataLoader(ScalpDataset(*splits['train'], train_transform), batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(ScalpDataset(*splits['val'], val_transform), batch_size=64, shuffle=False, num_workers=2)
test_dl = DataLoader(ScalpDataset(*splits['test'], val_transform), batch_size=64, shuffle=False, num_workers=2)

EPOCHS = 15
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
BEST_PATH = os.path.join(OUT_DIR, 'best.pt')

def run_eval(dl):
    model.eval(); preds, tgts = [], []
    with torch.no_grad():
        for x, y in dl:
            preds.extend(model(x.to(device)).argmax(1).cpu().numpy()); tgts.extend(y.numpy())
    preds, tgts = np.array(preds), np.array(tgts)
    return float((preds == tgts).mean() * 100), preds, tgts

best_val = 0.0
for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total += loss.item() * x.size(0)
    scheduler.step()
    val_acc, _, _ = run_eval(val_dl)
    print(f'epoch {epoch+1}/{EPOCHS} train_loss={total/len(train_dl.dataset):.4f} val_acc={val_acc:.2f}%')
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), BEST_PATH)
print('Best validation accuracy:', round(best_val, 2))

In [ ]:
# 7. Final evaluation on the held-out TEST split (best-val checkpoint)
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(BEST_PATH, map_location=device))
test_acc, preds, tgts = run_eval(test_dl)
print(f'TEST ACCURACY: {test_acc:.2f}%  (target 95%: {"MET" if test_acc >= 95 else "NOT MET"})')
print(classification_report(tgts, preds, target_names=CLASSES, zero_division=0))
cm = confusion_matrix(tgts, preds)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=60, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center')
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix.png'), dpi=150); plt.show()

In [ ]:
# 8. Export ONNX + class list to Drive
import json
model.eval()
ONNX_PATH = os.path.join(OUT_DIR, 'follica_mobilenetv4.onnx')
torch.onnx.export(model, torch.randn(1, 3, 224, 224).to(device), ONNX_PATH, opset_version=14,
                  do_constant_folding=True, input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
json.dump({'classes': CLASSES, 'architecture': ARCH, 'test_accuracy': test_acc, 'val_accuracy': best_val},
          open(os.path.join(OUT_DIR, 'follica_classes.json'), 'w'), indent=2)
print('Saved to', OUT_DIR, '| ONNX size MB:', round(os.path.getsize(ONNX_PATH) / 1048576, 2))